# End-to-End Regression Pipeline — California Housing (Assignment)

## Goal
Build a complete, production-ready regression pipeline on the **California Housing** dataset and save it as a deployable artifact.

Each step below tells you *what* to implement and *why* it matters. Write the code yourself in the empty cells. When stuck, check `01_regression_end_to_end.ipynb` for the worked solution.

## Dataset
`sklearn.datasets.fetch_california_housing(as_frame=True)` — 20,640 California districts, 8 numeric features, target = median house value (in $100k).

## Deliverable
By the end you should have:
- `artifacts_regression/model.pkl` and `artifacts_regression/model.joblib`
- `artifacts_regression/metadata.json` — versions, metrics, feature names
- A working reload + smoke test

---

## Step 0: Imports

All libraries you'll need are pre-loaded below — just run the cell.

**Why up front:** if a missing import would fail mid-notebook, you'd lose progress.

In [1]:
import os, json, pickle, joblib, sys
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import sklearn
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42
print('Imports OK')

Imports OK


## Step 1: Define the problem

**Task:** in a markdown cell below, answer:
1. What are we predicting? (regression vs classification)
2. Who would use it?
3. Which metric matters most? (RMSE, MAE, R²?)
4. What's the acceptance criterion?
5. Any data constraints?

**Why:** skipping this step is the #1 reason ML projects fail.

**YOUR ANSWERS HERE**
1. 
2. 
3. 
4. 
5. 

## Step 2: Load the data

Dataset is pre-loaded below. Just run the cell to inspect.

In [2]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
df.rename(columns={'MedHouseVal': 'target'}, inplace=True)
print(f'Shape: {df.shape}')
df.head()

Shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## Step 3: Exploratory Data Analysis

**Task:** check
1. column types (`.dtypes`)
2. missing values per column
3. summary statistics (`.describe()`)
4. target distribution (histogram) — is it skewed? Compare to `np.log1p(target)`
5. correlation heatmap — which features correlate with target?

**Why:** EDA tells you what preprocessing the data needs — you don't guess, you let the data tell you.

In [3]:
# Types, missing, describe
# YOUR CODE HERE


In [4]:
# Target distribution + log transform comparison
# YOUR CODE HERE


In [5]:
# Correlation heatmap
# YOUR CODE HERE


## Step 4: Clean & engineer features

**Task:**
1. Drop any duplicate rows.
2. Create at least 3 domain features. Suggestions: `bedrooms_ratio = AveBedrms / AveRooms`, `rooms_per_person = AveRooms / AveOccup`, `log_population = log1p(Population)`.
3. Inspect rows where target is at the cap (5.0) — note for later (likely censored).

**Why before splitting?** Cleaning + deterministic feature engineering is fine before splitting. Fitting transformers (imputer, scaler) is NOT — that goes inside a Pipeline.

In [6]:
# YOUR CODE HERE


## Step 5: Train / Test Split — BEFORE any fitting

**Task:** split into train/test (80/20) with `random_state=RANDOM_STATE`. Print shapes.

**Why now?** Everything from this point on (imputer, scaler, model selection, tuning) can leak test info if done on the full dataset.

In [7]:
# YOUR CODE HERE


## Step 6: Build the preprocessing Pipeline

**Task:** build a `ColumnTransformer` containing a numeric pipeline of `SimpleImputer(strategy='median')` → `StandardScaler()`. Don't fit it yet — Pipelines fit inside CV.

**Why a Pipeline?**
1. Prevents leakage during CV.
2. Reproducibility — one object holds the whole flow.
3. Same transformations applied at train and inference time.

In [8]:
# YOUR CODE HERE


## Step 7: Baseline model

**Task:** wrap `DummyRegressor(strategy='mean')` inside a Pipeline with the preprocessor. Fit on train, predict on test, report RMSE / MAE / R².

**Why a baseline?** Defines the bar to beat. If your fancy model can't beat predict-the-mean, something is wrong.

In [9]:
# YOUR CODE HERE


## Step 8: Try several models with cross-validation

**Task:** create at least 4 candidate models — e.g., `LinearRegression`, `Ridge`, `RandomForestRegressor`, `GradientBoostingRegressor`. Wrap each in a Pipeline with the preprocessor. Score each with 5-fold CV using `scoring='neg_root_mean_squared_error'`. Print a leaderboard sorted by mean CV RMSE.

**Why try multiple?** Linear vs tree models capture different patterns. A quick bake-off tells you which family suits the data without wasting time tuning the wrong one.

In [10]:
# YOUR CODE HERE


## Step 9: Hyperparameter tuning

**Task:** pick the best CV model. Use `GridSearchCV` to tune its 2–3 most-impactful hyperparameters. For Gradient Boosting, that's `n_estimators`, `learning_rate`, `max_depth`. Use `cv=3` and `scoring='neg_root_mean_squared_error'`.

**Tip:** when tuning a model inside a Pipeline, prefix grid keys with `model__` (e.g., `model__n_estimators`).

**Why?** Default hyperparameters are rarely optimal. Even a small grid usually buys 1–5% improvement.

In [11]:
# YOUR CODE HERE


## Step 10: Final evaluation on the test set (ONCE)

**Task:** predict on `X_test` with `grid.best_estimator_`. Report RMSE (also in dollars), MAE, R².

**Why ONCE?** Reporting CV scores as if they were test scores is a common mistake. The test set is sacred — touch it once at the end.

In [12]:
# YOUR CODE HERE


## Step 11: Inspect errors

**Task:** plot two charts side by side:
1. Predicted vs Actual scatter (with y=x diagonal).
2. Residuals vs Predicted.

**Why:** look for *patterns* in residuals. Curve → underfit; fan-shape → heteroscedasticity; cluster at the cap → censored target.

In [13]:
# YOUR CODE HERE


## Step 12: Explain the model

**Task:** compute `permutation_importance` on the test set with `n_repeats=10`. Plot a horizontal bar chart of mean importance per feature.

**Why permutation over MDI?** Permutation is unbiased and model-agnostic — it doesn't favor high-cardinality features the way tree impurity does.

In [14]:
# YOUR CODE HERE


## Step 13: Save the artifact

**Task:**
1. Create folder `artifacts_regression/`.
2. Save the final pipeline with both `pickle.dump` and `joblib.dump(..., compress=3)`.
3. Build a `metadata` dict containing: model_name, version, trained_at (ISO timestamp), feature_names, target_name, metrics (rmse/mae/r2), best_params, env (python/sklearn/numpy/pandas versions). Save to `metadata.json`.
4. Print the file sizes of pickle vs joblib — joblib should be smaller.

**Why metadata?** Six months from now you'll thank yourself. Versions, training date, feature names, and metrics are essential for debugging and audit.

In [15]:
# YOUR CODE HERE


## Step 14: Reload + smoke test

**Task:** load `model.joblib`. Take one row from `X_test`, predict, print predicted vs actual in dollars.

**Why:** confirms the artifact actually works end-to-end. Catches issues like missing dependencies or pipeline bugs that only show up at load time.

In [16]:
# YOUR CODE HERE


---
## Self-check questions

Once you've finished, write 1–2 sentence answers to these in markdown cells below.

1. Why didn't we fit the `StandardScaler` on the whole dataset before splitting?
2. Your linear model and your tree model give wildly different errors. Which would you trust more, and why?
3. The target has a cap at 5.0. What does this mean for your residual plot, and how would you handle it in production?
4. If RMSE is 0.5 (in $100k units) and MAE is 0.35 — what does the gap tell you about the error distribution?
5. You shipped the model. Six months later predictions look odd. What do you check first?

**YOUR ANSWERS**
1. 
2. 
3. 
4. 
5. 